In [ ]:
from rapidfuzz import process, fuzz

ROLE_BOOK_MAPPING = {
    "AI/ML Engineer": [...],
    "Data Scientist": [...],
    "Applied ML Engineer": [...],
    "Advanced ML Researcher": [...]
}

def get_closest_role(
    input_role: str,
    threshold: int = 70
):
    input_lower = input_role.lower()
    input_words = set(input_lower.split())

    role_book_maps_lower = [role.lower() for role in ROLE_BOOK_MAPPING.keys()]

    match = process.extractOne(
        input_lower,
        role_book_maps_lower,
        scorer=fuzz.WRatio
    )

    if match is None:
        return None

    matched_role, score, _ = match

    if len(input_words) < 2 and score < 90:
        return None
    
    # Additional check: at least 1 word from input appears in matched role
    matched_words = set(matched_role.split())
    if not input_words.intersection(matched_words):
        print(f"[Role Match] No word overlap: '{input_role}' vs '{matched_role}'")
        return None

    print(f"[Role Match] {matched_role} (score={score})")

    if score < threshold:
        return None

    return matched_role

In [15]:
get_closest_role("Software Engineer")

[Role Match] ai/ml engineer (score=69.09090909090908)


'ai/ml engineer'

In [6]:
from backend.agents.interview_question_agent import InterviewPlannerAgent


test_state = {
    "job_role": "Senior Backend Engineer (Distributed Systems)",
    "years_of_experience": 6,
    "extracted_skills": [
        "Python", 
        "Go", 
        "PostgreSQL", 
        "Redis", 
        "Kafka", 
        "Docker", 
        "Kubernetes", 
        "gRPC"
    ],
    "extracted_domains": [
        "FinTech", 
        "Scalable Architecture", 
        "Event-Driven Microservices"
    ],
    "resume_text": """
    Senior Software Engineer at FinTech Corp (2022 - Present)
    - Designed and implemented a high-throughput transaction processing engine handling 5,000+ RPS using Go and Kafka.
    - Optimized database performance, reducing PostgreSQL p99 latency by 40% through connection pooling and strategic indexing.
    - Migrated monolithic billing service into gRPC-based microservices deployed on AWS EKS.
    
    Software Engineer at DataScale Labs (2020 - 2022)
    - Built real-time analytics dashboards using Python, FastAPI, and Redis caching layers.
    - Maintained CI/CD pipelines and managed containerized deployments via Docker.
    """
}

test_state2 = {
    "job_role": "Data Engineer (ML Platform & Analytics)",
    "years_of_experience": 3,
    "extracted_skills": [
        "SQL",
        "Python",
        "Apache Spark",
        "AWS Redshift",
        "Airflow",
        "Terraform",
        "Snowflake"
    ],
    "extracted_domains": [
        "E-Commerce",
        "Big Data Engineering",
        "MLOps Pipeline Automation"
    ]
    # "resume_text" is intentionally omitted to test fallback behavior
}

test_state_intern = {
    "job_role": "Data Science & ML Intern",
    "years_of_experience": 0,
    "extracted_skills": [
        "Python",
        "PyTorch",
        "FastAPI",
        "Scikit-Learn",
        "Pandas",
        "Git"
    ],
    "extracted_domains": [
        "Predictive Modeling",
        "Computer Vision (Basic)",
        "Model Deployment"
    ],
}


In [9]:
agent = InterviewPlannerAgent(name="InterviewPlanner")
plan = await agent.execute(test_state_intern)

2026-05-20 16:29:18.561 | INFO     | backend.core.agent_base:get_completion:50 - [InterviewPlanner] Sending messages to GROQ/META-LLAMA/LLAMA-4-SCOUT-17B-16E-INSTRUCT:
2026-05-20 16:29:18.562 | DEBUG    | backend.core.agent_base:get_completion:52 -   system: 
You are an expert technical interviewer.

Generate a realistic interview roadmap.

Rules:

- Generate diverse interview topics.
- Maintain natural interview flow.
- Topics should support follow-up questions.
- Progress from easier to harder concepts.
- Avoid repetitive semantic areas.
- Generate retrieval-friendly search queries.
- Cover both theory and practical systems.
- Prefer realistic industry interview progression.

2026-05-20 16:29:18.563 | DEBUG    | backend.core.agent_base:get_completion:52 -   user: 
Role:
Data Science & ML Intern

Years of Experience:
0

Skills:
['Python', 'PyTorch', 'FastAPI', 'Scikit-Learn', 'Pandas', 'Git']

Domains:
['Predictive Modeling', 'Computer Vision (Basic)', 'Model Deployment']

Generate a 

Pydantic Schema Provided


2026-05-20 16:29:20.638 | INFO     | backend.core.agent_base:call_model:151 - [InterviewPlanner] Received structured response: namespace(content='{"role": "Data Science & ML Intern", "topics": [{"topic": "Python", "retrieval_query": "Python programming language", "subtopics": ["Basic syntax", "Data structures", "File handling"]}, {"topic": "Predictive Modeling", "retrieval_query": "Predictive modeling with scikit-learn", "subtopics": ["Supervised learning", "Unsupervised learning", "Model evaluation"]}, {"topic": "Model Deployment", "retrieval_query": "Deploying machine learning models with FastAPI", "subtopics": ["API design", "Model serving", "Testing and validation"]}, {"topic": "Computer Vision (Basic)", "retrieval_query": "Introduction to computer vision with PyTorch", "subtopics": ["Image processing", "Object detection", "Image classification"]}, {"topic": "Data Science", "retrieval_query": "Data science with Pandas and Scikit-Learn", "subtopics": ["Data cleaning and preprocessin

In [10]:
print(plan.model_dump_json(indent=4))

{
    "role": "Data Science & ML Intern",
    "topics": [
        {
            "topic": "Python",
            "retrieval_query": "Python programming language",
            "subtopics": [
                "Basic syntax",
                "Data structures",
                "File handling"
            ]
        },
        {
            "topic": "Predictive Modeling",
            "retrieval_query": "Predictive modeling with scikit-learn",
            "subtopics": [
                "Supervised learning",
                "Unsupervised learning",
                "Model evaluation"
            ]
        },
        {
            "topic": "Model Deployment",
            "retrieval_query": "Deploying machine learning models with FastAPI",
            "subtopics": [
                "API design",
                "Model serving",
                "Testing and validation"
            ]
        },
        {
            "topic": "Computer Vision (Basic)",
            "retrieval_query": "Introduction to c